In [1]:
# ============================================================
# CNC OVERTIME MINIMIZATION HEURISTIC
# Matematiksel modele uygun revize edilmiş versiyon
#
# OBJECTIVE : minimize Σm OTm  (toplam fazla mesai dakikası)
#
# HARD CONSTRAINTS (ihlal eden çözümler reddedilir):
#   C1 - Makine çakışması yok
#   C2 - Op10 → Op20 öncelik sırası korunur
#   C3 - Gecikme limiti aşılmaz (TARDINESS_LIMIT_DAYS)
#   C4 - Her iş tam adette üretilir (quantity conservation)
#   C5 - Setup süreleri çizelgeye dahil edilir
#   C6 - Makine-grup ataması eligible gruplar içinden yapılır
#
# REVISION: Warm start export eklendi.
#   Son bölümde export_warm_start() çağrılır.
#   Çıktı: heuristic_warm_start.json → Gurobi exact model bunu
#   model.optimize() öncesinde .Start olarak yükler.
# ============================================================

import json
import math
import random
import copy
import re
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timedelta, time
from collections import defaultdict

import pandas as pd
import numpy as np

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import plotly.graph_objects as go

# ============================================================
# 0. CONFIGURATION
# ============================================================

BASE_DIR = Path("C:\\Users\\emrec\\Documents\\parallel_scheduling\\heuristicandexact")

WARM_START_FILE = BASE_DIR / "heuristic_warm_start.json"


def find_input_file(patterns, description):
    for pattern in patterns:
        matches = sorted(BASE_DIR.glob(pattern))
        if matches:
            return matches[0]
    available = "\n".join([p.name for p in sorted(BASE_DIR.glob("*.xlsx"))])
    raise FileNotFoundError(
        f"Could not find {description}.\n"
        f"Searched patterns: {patterns}\n"
        f"Available Excel files:\n{available}"
    )


SHIPMENT_FILE = find_input_file(
    ["492-güncel sevkiyat*.xlsx", "492*güncel*sevkiyat*.xlsx", "*sevkiyat*.xlsx"],
    "shipment file"
)
SDST_FILE = find_input_file(["SDST*.xlsx", "*SDST*.xlsx"], "SDST file")
MACHINE_GROUP_FILE = find_input_file(
    ["machine_group_data*.xlsx", "*machine*group*.xlsx"],
    "machine group file"
)

OUTPUT_FILE = BASE_DIR / "optimized_revised_model.xlsx"
OUTPUT_VERIFICATION_FILE = BASE_DIR / "heuristic_constraint_verification_revised.xlsx"
OUTPUT_GANTT_HTML_FILE = BASE_DIR / "heuristic_gantt_revised.html"

print("Using files:")
print("SHIPMENT_FILE      =", SHIPMENT_FILE)
print("SDST_FILE          =", SDST_FILE)
print("MACHINE_GROUP_FILE =", MACHINE_GROUP_FILE)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PLANNING_START_HOUR = 7
DUE_TIME_HOUR = 17
OVERTIME_START = time(17, 0)
OVERTIME_END   = time(21, 0)
SHIPMENT_STYLE_OVERTIME = True

TARDINESS_LIMIT_DAYS = 1.5

# Split settings
ALLOW_JOB_SPLITTING  = True
MAX_SPLITS           = 2
MIN_SPLIT_QTY        = 100

# Setup settings
INITIAL_SETUP_MIN          = 10
SAME_DIAM_SETUP_MIN        = 5
DEFAULT_DIFF_DIAM_SETUP_MIN = 20

INFEASIBLE_PENALTY = float("inf")

# SA settings
SA_ITERATIONS      = 900
START_TEMPERATURE  = 250.0
END_TEMPERATURE    = 1.0

# Local search settings
LOCAL_SEARCH_ITERATIONS = 180


# ============================================================
# 1. HELPER FUNCTIONS
# ============================================================

def normalize_machine_name(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    s = s.replace(" ", "")
    s = s.replace(",", ".")
    s = s.replace("O", "0")
    s = s.replace("T3.", "T.3.")
    if re.match(r"^T3\d+", s):
        s = s.replace("T3", "T.3.", 1)
    return s


def excel_serial_to_datetime_date(x):
    if pd.isna(x):
        return None
    if isinstance(x, datetime):
        return x.date()
    if isinstance(x, (int, float, np.integer, np.floating)):
        return (datetime(1899, 12, 30) + timedelta(days=float(x))).date()
    parsed = pd.to_datetime(x, errors="coerce")
    if pd.isna(parsed):
        return None
    return parsed.date()


def excel_time_to_timedelta(x):
    if pd.isna(x):
        return None
    if isinstance(x, timedelta):
        return x
    if isinstance(x, datetime):
        return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, time):
        return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, (int, float, np.integer, np.floating)):
        return timedelta(days=float(x))
    parsed = pd.to_datetime(str(x), errors="coerce")
    if pd.isna(parsed):
        return None
    return timedelta(hours=parsed.hour, minutes=parsed.minute, seconds=parsed.second)


def combine_excel_date_time(date_value, time_value):
    d  = excel_serial_to_datetime_date(date_value)
    td = excel_time_to_timedelta(time_value)
    if d is None or td is None:
        return None
    return datetime.combine(d, time(0, 0)) + td


def minutes_between(start_dt, end_dt):
    if end_dt <= start_dt:
        end_dt = end_dt + timedelta(days=1)
    return (end_dt - start_dt).total_seconds() / 60.0


def overlap_minutes(a_start, a_end, b_start, b_end):
    latest_start  = max(a_start, b_start)
    earliest_end  = min(a_end,   b_end)
    return max(0.0, (earliest_end - latest_start).total_seconds() / 60.0)


def overtime_overlap_minutes(start_dt, end_dt):
    if end_dt <= start_dt:
        return 0.0
    total = 0.0
    current_day = start_dt.date()
    last_day    = end_dt.date()
    while current_day <= last_day:
        ot_start = datetime.combine(current_day, OVERTIME_START)
        ot_end   = datetime.combine(current_day, OVERTIME_END)
        total   += overlap_minutes(start_dt, end_dt, ot_start, ot_end)
        current_day = current_day + timedelta(days=1)
    return total


def next_overtime_end(dt):
    ot_s = datetime.combine(dt.date(), OVERTIME_START)
    ot_e = datetime.combine(dt.date(), OVERTIME_END)
    if ot_s <= dt < ot_e:
        return ot_e
    return dt


def align_start_to_shipment_calendar(dt):
    if not SHIPMENT_STYLE_OVERTIME:
        return dt
    return next_overtime_end(dt)


def parse_overtime_text(value):
    if pd.isna(value):
        return 0.0
    s = str(value).strip().lower()
    if not s:
        return 0.0
    hours = 0; minutes = 0
    h = re.search(r'(\d+)\s*saat', s)
    m = re.search(r'(\d+)\s*dk',   s)
    if h: hours   = int(h.group(1))
    if m: minutes = int(m.group(1))
    if not h and not m:
        nums = re.findall(r'\d+', s)
        if nums:
            minutes = int(nums[0])
    return float(hours * 60 + minutes)


def mode_or_first(series, default=None):
    s = series.dropna()
    if len(s) == 0:
        return default
    m = s.mode()
    if len(m) > 0:
        return m.iloc[0]
    return s.iloc[0]


# ============================================================
# 2. DATA CLASSES
# ============================================================

@dataclass
class Job:
    job_id: str
    part_no: str
    due_date: datetime
    quantity: int
    diameter: float
    eligible_groups_op10: list
    eligible_groups_op20: list


@dataclass
class TaskRecord:
    job_id: str
    part_no: str
    split_id: int
    operation: int
    quantity: int
    diameter: float
    group: int
    machine: str
    start: datetime
    finish: datetime
    processing_min: float
    setup_min: float
    overtime_min: float


# ============================================================
# 3. LOAD INPUT DATA
# ============================================================

def load_shipment_operations(path):
    raw = pd.read_excel(path, sheet_name=0, header=5, usecols="D:N", engine="openpyxl")
    raw.columns = [str(c).strip().replace(":", "") for c in raw.columns]
    raw = raw.dropna(how="all")

    rename_map = {
        "Tarih": "date",
        "Başlangıç Saat": "start_time",
        "Bitiş Saat": "finish_time",
        "Parça No": "part_no",
        "Makine No": "machine",
        "CNC-1 operasyonu(piston)": "op10_flag",
        "CNC-2 operasyonu (saplama)": "op20_flag",
        "Adet": "quantity",
        "Çap": "diameter",
        "Makine Grubu": "machine_group",
        "Fazla mesai": "overtime_text",
    }
    raw = raw.rename(columns=rename_map)

    required = ["date", "start_time", "finish_time", "part_no", "machine",
                "quantity", "diameter", "machine_group"]
    for col in required:
        if col not in raw.columns:
            raise ValueError(f"Missing expected column: {col}")

    raw = raw[raw["part_no"].notna()].copy()
    raw["machine"]       = raw["machine"].apply(normalize_machine_name)
    raw["part_no"]       = raw["part_no"].astype(int).astype(str)
    raw["quantity"]      = pd.to_numeric(raw["quantity"],      errors="coerce").fillna(0).astype(int)
    raw["diameter"]      = pd.to_numeric(raw["diameter"],      errors="coerce")
    raw["machine_group"] = pd.to_numeric(raw["machine_group"], errors="coerce").astype("Int64")

    raw["operation"] = np.where(
        raw.get("op10_flag").notna(), 10,
        np.where(raw.get("op20_flag").notna(), 20, np.nan)
    )
    raw = raw[raw["operation"].notna()].copy()
    raw["operation"] = raw["operation"].astype(int)

    raw["start_dt"]    = [combine_excel_date_time(d, t) for d, t in zip(raw["date"], raw["start_time"])]
    raw["finish_dt"]   = [combine_excel_date_time(d, t) for d, t in zip(raw["date"], raw["finish_time"])]
    raw["duration_min"] = [minutes_between(s, f) for s, f in zip(raw["start_dt"], raw["finish_dt"])]
    raw["unit_min"]    = raw["duration_min"] / raw["quantity"].replace(0, np.nan)

    return raw


def load_orders_from_sayfa2(path, use_shipped_quantity=False):
    df = pd.read_excel(path, sheet_name=1, header=None, engine="openpyxl")
    jobs = []
    current_date = None
    in_block     = False

    for _, row in df.iterrows():
        first_values = row.dropna().tolist()
        if len(first_values) == 0:
            continue

        possible_date = None
        for val in row.tolist():
            if pd.isna(val): continue
            if isinstance(val, (int, float, np.integer, np.floating)) and 40000 < float(val) < 60000:
                possible_date = excel_serial_to_datetime_date(val); break
            if isinstance(val, datetime):
                possible_date = val.date(); break

        if possible_date is not None:
            current_date = possible_date
            in_block = False
            continue

        row_text = " ".join([str(v).lower() for v in row.dropna().tolist()])
        if "sipariş" in row_text and "adet" in row_text:
            in_block = True
            continue

        if in_block and current_date is not None:
            part       = row.iloc[4] if len(row) > 4 else None
            order_qty  = row.iloc[5] if len(row) > 5 else None
            shipped_qty = row.iloc[6] if len(row) > 6 else None

            if pd.isna(part) or pd.isna(order_qty):
                continue

            try:
                part_no = str(int(part))
            except Exception:
                continue

            qty_source = shipped_qty if use_shipped_quantity and not pd.isna(shipped_qty) else order_qty
            qty    = int(round(float(qty_source)))
            due_dt = datetime.combine(current_date, time(DUE_TIME_HOUR, 0))
            job_id = f"{current_date.isoformat()}__{part_no}"
            jobs.append({"job_id": job_id, "part_no": part_no, "due_date": due_dt, "quantity": qty})

    orders = pd.DataFrame(jobs)
    if orders.empty:
        raise ValueError("Could not parse any order rows from Sayfa2.")

    before_n = len(orders)
    orders = (
        orders.groupby(["job_id", "part_no"], as_index=False)
        .agg(quantity=("quantity", "sum"), due_date=("due_date", "min"))
    )
    after_n = len(orders)
    if before_n != after_n:
        print(f"Duplicate order rows aggregated: {before_n} -> {after_n} unique jobs")

    return orders


def load_machine_groups(path):
    mg = pd.read_excel(path, sheet_name=0, engine="openpyxl")
    mg.columns = [str(c).strip() for c in mg.columns]
    mg = mg.dropna(subset=["Machine_number", "Group"]).copy()
    mg["Machine_number"] = mg["Machine_number"].apply(normalize_machine_name)
    mg["Group"]          = pd.to_numeric(mg["Group"], errors="coerce").astype(int)

    group_to_machines = defaultdict(list)
    machine_to_group  = {}
    for _, r in mg.iterrows():
        m = r["Machine_number"]
        g = int(r["Group"])
        group_to_machines[g].append(m)
        machine_to_group[m] = g

    return dict(group_to_machines), machine_to_group


def load_sdst(path):
    sdst = pd.read_excel(path, sheet_name=0, engine="openpyxl")
    sdst.columns = [str(c).strip() for c in sdst.columns]
    setup = {}
    for _, r in sdst.dropna(subset=["diam_from", "to_diam", "setup_time"]).iterrows():
        d1 = float(r["diam_from"])
        d2 = float(r["to_diam"])
        setup[(d1, d2)] = float(r["setup_time"])
    return setup


# ============================================================
# 4. BUILD MODEL DATA
# ============================================================

def build_problem_data(shipment_file, sdst_file, machine_group_file, use_shipped_quantity=False):
    ops    = load_shipment_operations(shipment_file)
    orders = load_orders_from_sayfa2(shipment_file, use_shipped_quantity=use_shipped_quantity)
    group_to_machines, machine_to_group = load_machine_groups(machine_group_file)
    setup_dict = load_sdst(sdst_file)

    part_diam = ops.groupby("part_no")["diameter"].agg(lambda x: mode_or_first(x, default=0)).to_dict()

    eligible = defaultdict(lambda: defaultdict(set))
    for _, r in ops.iterrows():
        eligible[r["part_no"]][int(r["operation"])].add(int(r["machine_group"]))

    unit_time = {}
    grouped   = ops.dropna(subset=["unit_min", "machine_group"]).groupby(
        ["part_no", "operation", "machine_group"]
    )
    for key, g in grouped:
        unit_time[(str(key[0]), int(key[1]), int(key[2]))] = float(g["unit_min"].median())

    fallback_part_op = ops.dropna(subset=["unit_min"]).groupby(
        ["part_no", "operation"])["unit_min"].median().to_dict()
    fallback_op  = ops.dropna(subset=["unit_min"]).groupby("operation")["unit_min"].median().to_dict()
    global_unit  = float(ops["unit_min"].dropna().median())

    jobs       = []
    all_groups = sorted(group_to_machines.keys())

    for _, r in orders.iterrows():
        part_no = str(r["part_no"])
        q       = int(r["quantity"])
        diam    = float(part_diam.get(part_no, ops["diameter"].dropna().median()))

        eg10 = sorted(list(eligible[part_no].get(10, set(all_groups))))
        eg20 = sorted(list(eligible[part_no].get(20, set(eg10 if eg10 else all_groups))))
        if not eg10: eg10 = all_groups
        if not eg20: eg20 = eg10

        jobs.append(Job(
            job_id=r["job_id"], part_no=part_no, due_date=r["due_date"],
            quantity=q, diameter=diam,
            eligible_groups_op10=eg10, eligible_groups_op20=eg20
        ))

    ops["shipment_file_overtime_min"] = ops.get(
        "overtime_text", pd.Series(dtype=object)
    ).apply(parse_overtime_text)
    baseline_overtime_min = float(ops["shipment_file_overtime_min"].sum())

    data = {
        "ops": ops,
        "baseline_overtime_min": baseline_overtime_min,
        "orders": orders,
        "jobs": jobs,
        "group_to_machines":  group_to_machines,
        "machine_to_group":   machine_to_group,
        "setup_dict":         setup_dict,
        "unit_time":          unit_time,
        "fallback_part_op":   fallback_part_op,
        "fallback_op":        fallback_op,
        "global_unit":        global_unit,
        "planning_start": datetime.combine(
            min([j.due_date.date() for j in jobs]),
            time(PLANNING_START_HOUR, 0)
        ),
    }
    return data


def get_unit_time(data, part_no, operation, group):
    key = (str(part_no), int(operation), int(group))
    if key in data["unit_time"]:
        return data["unit_time"][key]
    key2 = (str(part_no), int(operation))
    if key2 in data["fallback_part_op"]:
        return float(data["fallback_part_op"][key2])
    if int(operation) in data["fallback_op"]:
        return float(data["fallback_op"][int(operation)])
    return data["global_unit"]


def get_setup_time(data, from_diam, to_diam):
    if from_diam is None:
        return INITIAL_SETUP_MIN
    d1 = float(from_diam); d2 = float(to_diam)
    if abs(d1 - d2) < 1e-9:
        return SAME_DIAM_SETUP_MIN
    if (d1, d2) in data["setup_dict"]:
        return data["setup_dict"][(d1, d2)]
    return DEFAULT_DIFF_DIAM_SETUP_MIN


def common_eligible_groups(job):
    common = sorted(list(set(job.eligible_groups_op10).intersection(set(job.eligible_groups_op20))))
    if common:
        return common
    return sorted(job.eligible_groups_op10)


# ============================================================
# 5. SOLUTION REPRESENTATION
# ============================================================

def make_split_quantities(q):
    if not ALLOW_JOB_SPLITTING or q < 2 * MIN_SPLIT_QTY:
        return [int(q)]
    q1 = int(math.ceil(q / 2))
    q2 = int(q - q1)
    if q2 < MIN_SPLIT_QTY:
        return [int(q)]
    return [q1, q2]


def build_initial_solution(data):
    sol = {"splits": {}, "group": {}, "m10": {}, "m20": {}, "order": []}
    for job in data["jobs"]:
        qs     = make_split_quantities(job.quantity)
        sol["splits"][job.job_id] = qs
        groups = common_eligible_groups(job)
        for sid, _qty in enumerate(qs, start=1):
            key      = (job.job_id, sid)
            g        = random.choice(groups)
            machines = data["group_to_machines"].get(g, [])
            if not machines:
                raise ValueError(f"No machine found for group {g}.")
            sol["group"][key] = g
            sol["m10"][key]   = random.choice(machines)
            sol["m20"][key]   = random.choice(machines)
            sol["order"].append(key)

    jobs_by_id = {j.job_id: j for j in data["jobs"]}
    sol["order"].sort(key=lambda k: jobs_by_id[k[0]].due_date)
    return sol


def repair_solution(sol, data):
    jobs_by_id = {j.job_id: j for j in data["jobs"]}
    new_order  = []

    for job in data["jobs"]:
        qs = sol["splits"].get(job.job_id, [job.quantity])
        qs = [int(max(0, q)) for q in qs if q > 0]
        qs = qs[:MAX_SPLITS]

        if len(qs) > 1 and any(q < MIN_SPLIT_QTY for q in qs):
            qs = [job.quantity]

        diff = job.quantity - sum(qs)
        if len(qs) == 0:
            qs = [job.quantity]
        else:
            qs[-1] += diff

        if len(qs) > 1 and qs[-1] < MIN_SPLIT_QTY:
            qs = [job.quantity]

        sol["splits"][job.job_id] = qs
        groups = common_eligible_groups(job)

        for sid, _qty in enumerate(qs, start=1):
            key = (job.job_id, sid)
            if key not in sol["group"] or sol["group"][key] not in groups:
                sol["group"][key] = random.choice(groups)
            g        = sol["group"][key]
            machines = data["group_to_machines"].get(g, [])
            if not machines:
                g = random.choice(groups)
                sol["group"][key] = g
                machines = data["group_to_machines"][g]
            if sol["m10"].get(key) not in machines:
                sol["m10"][key] = random.choice(machines)
            if sol["m20"].get(key) not in machines:
                sol["m20"][key] = random.choice(machines)
            new_order.append(key)

    valid_keys = set(new_order)
    old_order  = []
    seen       = set()
    for k in sol.get("order", []):
        if k in valid_keys and k not in seen:
            old_order.append(k)
            seen.add(k)

    missing    = [k for k in new_order if k not in seen]
    sol["order"] = old_order + missing
    sol["order"] = [
        k for k in sol["order"]
        if k[0] in sol["splits"] and 1 <= int(k[1]) <= len(sol["splits"][k[0]])
    ]
    return sol


# ============================================================
# 6. DECODER / SCHEDULER
# ============================================================

def schedule_solution(sol, data):
    sol = repair_solution(copy.deepcopy(sol), data)
    jobs_by_id = {j.job_id: j for j in data["jobs"]}

    machine_available  = {}
    machine_last_diam  = {}
    for m in data["machine_to_group"].keys():
        machine_available[m] = data["planning_start"]
        machine_last_diam[m] = None

    records    = []
    finish_op10 = {}

    for key in sol["order"]:
        job_id, sid = key
        job   = jobs_by_id[job_id]
        qty   = sol["splits"][job_id][sid - 1]
        g     = sol["group"][key]
        m10   = sol["m10"][key]
        m20   = sol["m20"][key]

        unit10  = get_unit_time(data, job.part_no, 10, g)
        proc10  = qty * unit10
        setup10 = get_setup_time(data, machine_last_diam.get(m10), job.diameter)
        start10 = max(machine_available[m10], data["planning_start"]) + timedelta(minutes=setup10)
        start10 = align_start_to_shipment_calendar(start10)
        finish10 = start10 + timedelta(minutes=proc10)

        records.append(TaskRecord(
            job_id=job.job_id, part_no=job.part_no, split_id=sid,
            operation=10, quantity=qty, diameter=job.diameter,
            group=g, machine=m10, start=start10, finish=finish10,
            processing_min=proc10, setup_min=setup10,
            overtime_min=overtime_overlap_minutes(start10, finish10)
        ))
        machine_available[m10] = finish10
        machine_last_diam[m10] = job.diameter
        finish_op10[key]       = finish10

        unit20  = get_unit_time(data, job.part_no, 20, g)
        proc20  = qty * unit20
        setup20 = get_setup_time(data, machine_last_diam.get(m20), job.diameter)
        start20 = max(machine_available[m20], finish_op10[key]) + timedelta(minutes=setup20)
        start20 = align_start_to_shipment_calendar(start20)
        finish20 = start20 + timedelta(minutes=proc20)

        records.append(TaskRecord(
            job_id=job.job_id, part_no=job.part_no, split_id=sid,
            operation=20, quantity=qty, diameter=job.diameter,
            group=g, machine=m20, start=start20, finish=finish20,
            processing_min=proc20, setup_min=setup20,
            overtime_min=overtime_overlap_minutes(start20, finish20)
        ))
        machine_available[m20] = finish20
        machine_last_diam[m20] = job.diameter

    schedule_df = pd.DataFrame([r.__dict__ for r in records])
    if schedule_df.empty:
        raise ValueError("Schedule is empty.")
    return schedule_df


def add_split_count(schedule_df):
    df = schedule_df.copy()
    split_counts    = df.groupby("job_id")["split_id"].max().to_dict()
    df["split_count"] = df["job_id"].map(split_counts).astype(int)
    return df


def check_machine_overlaps(schedule_df, tolerance_seconds=1):
    df = schedule_df.copy()
    df["start"]  = pd.to_datetime(df["start"])
    df["finish"] = pd.to_datetime(df["finish"])
    overlaps = []
    for machine, g in df.sort_values(["machine", "start", "finish"]).groupby("machine"):
        rows = g.to_dict("records")
        for idx in range(len(rows) - 1):
            current = rows[idx]; nxt = rows[idx + 1]
            gap_sec = (nxt["start"] - current["finish"]).total_seconds()
            if gap_sec < -tolerance_seconds:
                overlaps.append({
                    "machine": machine,
                    "task_1": f"{current['part_no']} S{current['split_id']} Op{current['operation']}",
                    "task_1_finish": current["finish"],
                    "task_2": f"{nxt['part_no']} S{nxt['split_id']} Op{nxt['operation']}",
                    "task_2_start": nxt["start"],
                    "overlap_min": round(abs(gap_sec) / 60.0, 3),
                })
    return pd.DataFrame(overlaps)


# ============================================================
# 7. FEASIBILITY CHECK
# ============================================================

def check_feasibility(schedule_df, sol, data):
    violations = {}

    overlaps = check_machine_overlaps(schedule_df)
    if not overlaps.empty:
        violations["C1_machine_overlap"] = float(overlaps["overlap_min"].sum())

    for (job_id, split_id), g in schedule_df.groupby(["job_id", "split_id"]):
        op10 = g[g["operation"] == 10]
        op20 = g[g["operation"] == 20]
        if op10.empty or op20.empty:
            violations.setdefault("C2_precedence", 0)
            violations["C2_precedence"] += 1
            continue
        slack = (op20["start"].iloc[0] - op10["finish"].iloc[0]).total_seconds()
        if slack < -1e-3:
            violations.setdefault("C2_precedence", 0)
            violations["C2_precedence"] += abs(slack) / 60.0

    jobs_by_id          = {j.job_id: j for j in data["jobs"]}
    tardiness_limit_min = TARDINESS_LIMIT_DAYS * 24 * 60
    tard_violation      = 0.0
    schedule_df["finish"] = pd.to_datetime(schedule_df["finish"])
    for job_id, g in schedule_df[schedule_df["operation"] == 20].groupby("job_id"):
        job          = jobs_by_id[job_id]
        completion   = g["finish"].max()
        allowed      = job.due_date + timedelta(minutes=tardiness_limit_min)
        excess       = (completion - allowed).total_seconds() / 60.0
        if excess > 1e-3:
            tard_violation += excess
    if tard_violation > 0:
        violations["C3_tardiness_limit"] = tard_violation

    for job in data["jobs"]:
        qs = sol["splits"].get(job.job_id, [])
        if abs(sum(qs) - job.quantity) > 0:
            violations.setdefault("C4_quantity", 0)
            violations["C4_quantity"] += abs(sum(qs) - job.quantity)

    for job in data["jobs"]:
        groups = common_eligible_groups(job)
        for sid in range(1, len(sol["splits"].get(job.job_id, [])) + 1):
            key = (job.job_id, sid)
            g   = sol["group"].get(key)
            if g not in groups:
                violations.setdefault("C6_group_eligibility", 0)
                violations["C6_group_eligibility"] += 1

    is_feasible = len(violations) == 0
    return is_feasible, violations


# ============================================================
# 8. OBJECTIVE FUNCTION
# ============================================================

def evaluate_solution(sol, data):
    try:
        sched = schedule_solution(sol, data)
        sched = add_split_count(sched)
    except Exception as e:
        return INFEASIBLE_PENALTY, {"score": INFEASIBLE_PENALTY, "infeasible_reason": str(e)}

    is_feasible, violations = check_feasibility(sched, sol, data)

    if not is_feasible:
        return INFEASIBLE_PENALTY, {
            "score": INFEASIBLE_PENALTY,
            "feasible": False,
            "violations": violations,
            "total_overtime": float(sched["overtime_min"].sum()),
        }

    total_overtime = float(sched["overtime_min"].sum())
    score = total_overtime

    makespan    = (sched["finish"].max() - data["planning_start"]).total_seconds() / 60.0
    total_setup = float(sched["setup_min"].sum())

    jobs_by_id = {j.job_id: j for j in data["jobs"]}
    tardiness_limit_min  = TARDINESS_LIMIT_DAYS * 24 * 60
    total_tardiness      = 0.0
    max_tardiness_days   = 0.0

    for job_id, g in sched[sched["operation"] == 20].groupby("job_id"):
        job         = jobs_by_id[job_id]
        finish_dt   = g["finish"].max()
        tard_min    = max(0.0, (finish_dt - job.due_date).total_seconds() / 60.0)
        total_tardiness   += tard_min
        max_tardiness_days = max(max_tardiness_days, tard_min / 1440.0)

    metrics = {
        "score":                    score,
        "feasible":                 True,
        "violations":               {},
        "total_overtime":           total_overtime,
        "baseline_overtime_min":    data.get("baseline_overtime_min", 0.0),
        "overtime_improvement_min": data.get("baseline_overtime_min", 0.0) - total_overtime,
        "total_tardiness_from_due": total_tardiness,
        "max_tardiness_days":       max_tardiness_days,
        "tardiness_limit_days":     TARDINESS_LIMIT_DAYS,
        "total_tardiness_limit_violation": 0.0,
        "makespan":                 makespan,
        "total_setup":              total_setup,
        "total_overlap_min":        0.0,
        "overlap_count":            0,
    }
    return score, metrics


# ============================================================
# 9. NEIGHBORHOOD MOVES
# ============================================================

def get_overtime_task_candidates(sol, data, top_n=8):
    try:
        sched = schedule_solution(sol, data)
    except Exception:
        return []
    if sched.empty or "overtime_min" not in sched.columns:
        return []
    ot = sched[sched["overtime_min"] > 1e-6].copy()
    if ot.empty:
        return []
    ot = ot.sort_values("overtime_min", ascending=False).head(top_n)
    return [
        {
            "key":         (r["job_id"], int(r["split_id"])),
            "operation":   int(r["operation"]),
            "machine":     r["machine"],
            "overtime_min": float(r["overtime_min"]),
            "start":       r["start"],
            "finish":      r["finish"],
            "part_no":     r["part_no"],
        }
        for _, r in ot.iterrows()
    ]


def choose_overtime_candidate(sol, data):
    candidates = get_overtime_task_candidates(sol, data, top_n=6)
    if not candidates:
        return None
    weights = [max(1.0, c["overtime_min"]) for c in candidates]
    return random.choices(candidates, weights=weights, k=1)[0]


def move_key_to_position(order, key, new_pos):
    if key not in order:
        return order
    new_order = [k for k in order if k != key]
    new_pos   = max(0, min(int(new_pos), len(new_order)))
    new_order.insert(new_pos, key)
    return new_order


def targeted_overtime_mutation(sol, data):
    new  = copy.deepcopy(sol)
    cand = choose_overtime_candidate(new, data)
    if cand is None:
        return mutate_solution(new, data)

    key    = cand["key"]
    op     = cand["operation"]
    job_id, sid = key
    jobs_by_id  = {j.job_id: j for j in data["jobs"]}
    if job_id not in jobs_by_id:
        return repair_solution(new, data)
    job = jobs_by_id[job_id]

    move = random.choice([
        "insert_earlier",
        "swap_with_previous",
        "change_machine",
        "change_group",
        "reduce_split_qty",
        "diameter_neighbor_swap",
    ])

    if move == "insert_earlier" and key in new["order"]:
        pos = new["order"].index(key)
        if pos > 0:
            step = random.randint(1, min(pos, 5))
            new["order"] = move_key_to_position(new["order"], key, pos - step)

    elif move == "swap_with_previous" and key in new["order"]:
        pos = new["order"].index(key)
        if pos > 0:
            j = max(0, pos - random.randint(1, min(pos, 2)))
            new["order"][pos], new["order"][j] = new["order"][j], new["order"][pos]

    elif move == "change_machine":
        g        = new["group"].get(key)
        machines = data["group_to_machines"].get(g, [])
        if machines:
            if op == 10:
                cur  = new["m10"].get(key)
                alts = [m for m in machines if m != cur]
                if alts: new["m10"][key] = random.choice(alts)
            else:
                cur  = new["m20"].get(key)
                alts = [m for m in machines if m != cur]
                if alts: new["m20"][key] = random.choice(alts)

    elif move == "change_group":
        groups = common_eligible_groups(job)
        cur_g  = new["group"].get(key)
        alts   = [g for g in groups if g != cur_g and data["group_to_machines"].get(g)]
        if alts:
            g        = random.choice(alts)
            machines = data["group_to_machines"][g]
            new["group"][key] = g
            new["m10"][key]   = random.choice(machines)
            new["m20"][key]   = random.choice(machines)

    elif move == "reduce_split_qty":
        qs = list(new["splits"].get(job_id, [job.quantity]))
        if len(qs) == 2 and job.quantity >= 2 * MIN_SPLIT_QTY:
            idx   = sid - 1
            other = 1 - idx
            if 0 <= idx < 2 and qs[idx] > MIN_SPLIT_QTY:
                max_delta = min(qs[idx] - MIN_SPLIT_QTY, max(10, int(job.quantity * 0.15)))
                if max_delta > 0:
                    step  = random.choice([10, 25, 50, 75, 100])
                    delta = min(max_delta, step)
                    qs[idx]   -= delta
                    qs[other] += delta
                    if qs[0] >= MIN_SPLIT_QTY and qs[1] >= MIN_SPLIT_QTY:
                        new["splits"][job_id] = [int(qs[0]), int(qs[1])]

    elif move == "diameter_neighbor_swap" and key in new["order"]:
        pos    = new["order"].index(key)
        window = list(range(max(0, pos - 4), min(len(new["order"]), pos + 5)))
        window = [w for w in window if w != pos]
        if window:
            jpos = random.choice(window)
            new["order"][pos], new["order"][jpos] = new["order"][jpos], new["order"][pos]

    return repair_solution(new, data)


def mutate_solution(sol, data):
    new        = copy.deepcopy(sol)
    jobs_by_id = {j.job_id: j for j in data["jobs"]}

    move = random.choice([
        "swap_order",
        "change_machine10",
        "change_machine20",
        "change_group",
        "change_split_ratio",
        "toggle_split",
    ])

    if move == "swap_order" and len(new["order"]) >= 2:
        i, j = random.sample(range(len(new["order"])), 2)
        new["order"][i], new["order"][j] = new["order"][j], new["order"][i]

    elif move == "change_machine10" and new["order"]:
        key      = random.choice(new["order"])
        g        = new["group"][key]
        machines = data["group_to_machines"].get(g, [])
        if machines: new["m10"][key] = random.choice(machines)

    elif move == "change_machine20" and new["order"]:
        key      = random.choice(new["order"])
        g        = new["group"][key]
        machines = data["group_to_machines"].get(g, [])
        if machines: new["m20"][key] = random.choice(machines)

    elif move == "change_group" and new["order"]:
        key      = random.choice(new["order"])
        job      = jobs_by_id[key[0]]
        groups   = common_eligible_groups(job)
        if groups:
            g        = random.choice(groups)
            machines = data["group_to_machines"].get(g, [])
            if machines:
                new["group"][key] = g
                new["m10"][key]   = random.choice(machines)
                new["m20"][key]   = random.choice(machines)

    elif move == "change_split_ratio":
        job = random.choice(data["jobs"])
        qs  = new["splits"].get(job.job_id, [job.quantity])
        if len(qs) == 2 and job.quantity >= 2 * MIN_SPLIT_QTY:
            delta = random.randint(-max(1, job.quantity // 10), max(1, job.quantity // 10))
            q1 = qs[0] + delta; q2 = job.quantity - q1
            if q1 >= MIN_SPLIT_QTY and q2 >= MIN_SPLIT_QTY:
                new["splits"][job.job_id] = [q1, q2]

    elif move == "toggle_split":
        job = random.choice(data["jobs"])
        qs  = new["splits"].get(job.job_id, [job.quantity])
        if len(qs) == 1 and ALLOW_JOB_SPLITTING and job.quantity >= 2 * MIN_SPLIT_QTY:
            q1 = int(math.ceil(job.quantity / 2)); q2 = int(job.quantity - q1)
            if q2 >= MIN_SPLIT_QTY:
                new["splits"][job.job_id] = [q1, q2]
                for s in [1, 2]:
                    k        = (job.job_id, s)
                    groups   = common_eligible_groups(job)
                    g        = random.choice(groups)
                    machines = data["group_to_machines"][g]
                    new["group"][k] = g
                    new["m10"][k]   = random.choice(machines)
                    new["m20"][k]   = random.choice(machines)
        elif len(qs) == 2:
            new["splits"][job.job_id] = [job.quantity]

    return repair_solution(new, data)


# ============================================================
# 10. BEST-INSERTION LOCAL SEARCH
# ============================================================

def best_insertion_for_overtime_task(sol, data, max_candidates=4, max_trials=600):
    base       = repair_solution(copy.deepcopy(sol), data)
    base_score, base_metrics = evaluate_solution(base, data)
    best       = copy.deepcopy(base)
    best_score = base_score
    best_metrics = base_metrics

    candidates = get_overtime_task_candidates(base, data, top_n=max_candidates)
    if not candidates:
        return base, base_metrics, False

    tried = 0
    for cand in candidates:
        key = cand["key"]
        op  = int(cand["operation"])
        if key not in base.get("order", []):
            continue

        g        = base["group"].get(key)
        machines = list(data["group_to_machines"].get(g, []))
        if not machines:
            continue

        current_machine   = base["m10"].get(key) if op == 10 else base["m20"].get(key)
        machine_candidates = [current_machine] + [m for m in machines if m != current_machine]
        machine_candidates = [m for m in machine_candidates if m is not None]

        old_order_without_key = [k for k in base["order"] if k != key]
        for pos in range(len(old_order_without_key) + 1):
            if tried >= max_trials: break
            for mach in machine_candidates:
                if tried >= max_trials: break
                new = copy.deepcopy(base)
                new_order = list(old_order_without_key)
                new_order.insert(pos, key)
                new["order"] = new_order
                if op == 10: new["m10"][key] = mach
                else:        new["m20"][key] = mach
                new   = repair_solution(new, data)
                score, metrics = evaluate_solution(new, data)
                tried += 1
                if score < best_score and metrics.get("feasible", False):
                    best       = new
                    best_score = score
                    best_metrics = metrics
        if tried >= max_trials:
            break

    improved = best_score < base_score - 1e-6
    return best, best_metrics, improved


def best_insertion_post_search(best, data, rounds=8):
    current       = copy.deepcopy(best)
    current_score, current_metrics = evaluate_solution(current, data)
    history       = []
    for r in range(1, rounds + 1):
        candidate, metrics, improved = best_insertion_for_overtime_task(
            current, data, max_candidates=2, max_trials=180
        )
        if not improved: break
        cand_score = metrics["score"]
        if cand_score < current_score:
            current        = candidate
            current_score  = cand_score
            current_metrics = metrics
            history.append({"iteration": r, "best_score": current_score, **current_metrics})
        else:
            break
    return current, current_metrics, pd.DataFrame(history)


def split_quantity_fine_tuning_post_search(best, data, rounds=25, top_candidates=8):
    current       = repair_solution(copy.deepcopy(best), data)
    current_score, current_metrics = evaluate_solution(current, data)
    jobs_by_id    = {j.job_id: j for j in data["jobs"]}
    history       = []
    step_sizes    = [10, 25, 50, 75, 100, 150, 200]

    for r in range(1, rounds + 1):
        improved         = False
        best_round       = copy.deepcopy(current)
        best_round_score = current_score
        best_round_metrics = current_metrics

        candidates = get_overtime_task_candidates(current, data, top_n=top_candidates)
        if not candidates: break
        candidates = sorted(candidates, key=lambda c: c["overtime_min"], reverse=True)

        for cand in candidates:
            job_id, sid = cand["key"]
            job = jobs_by_id.get(job_id)
            if job is None: continue
            qs  = list(current["splits"].get(job_id, [job.quantity]))
            if len(qs) != 2 or sid not in (1, 2): continue

            idx   = sid - 1
            other = 1 - idx

            for sign_idx, sign_other in [(-1, +1), (+1, -1)]:
                for step in step_sizes:
                    new_qs = list(qs)
                    new_qs[idx]   += sign_idx * step
                    new_qs[other] += sign_other * step
                    if new_qs[idx] < MIN_SPLIT_QTY or new_qs[other] < MIN_SPLIT_QTY: continue
                    diff = job.quantity - sum(new_qs)
                    new_qs[other] += diff
                    if new_qs[idx] < MIN_SPLIT_QTY or new_qs[other] < MIN_SPLIT_QTY: continue

                    trial = copy.deepcopy(current)
                    trial["splits"][job_id] = [int(new_qs[0]), int(new_qs[1])]
                    trial = repair_solution(trial, data)
                    score, metrics = evaluate_solution(trial, data)

                    if score < best_round_score - 1e-6 and metrics.get("feasible", False):
                        best_round         = trial
                        best_round_score   = score
                        best_round_metrics = metrics
                        improved           = True

        if not improved: break
        current         = best_round
        current_score   = best_round_score
        current_metrics = best_round_metrics
        history.append({"iteration": r, "best_score": current_score, **current_metrics})

    return current, current_metrics, pd.DataFrame(history)


# ============================================================
# 11. SIMULATED ANNEALING + LOCAL SEARCH
# ============================================================

def simulated_annealing(data):
    current       = build_initial_solution(data)
    current_score, current_metrics = evaluate_solution(current, data)

    attempts = 0
    while current_score == INFEASIBLE_PENALTY and attempts < 500:
        current = repair_solution(build_initial_solution(data), data)
        current_score, current_metrics = evaluate_solution(current, data)
        attempts += 1

    best         = copy.deepcopy(current)
    best_score   = current_score
    best_metrics = current_metrics
    history      = []

    for it in range(1, SA_ITERATIONS + 1):
        temp = START_TEMPERATURE * ((END_TEMPERATURE / START_TEMPERATURE) ** (it / SA_ITERATIONS))

        candidate = (
            targeted_overtime_mutation(current, data)
            if random.random() < 0.65
            else mutate_solution(current, data)
        )
        cand_score, cand_metrics = evaluate_solution(candidate, data)

        if cand_score == INFEASIBLE_PENALTY:
            continue

        delta  = cand_score - current_score
        accept = (delta < 0) or (
            current_score != INFEASIBLE_PENALTY
            and random.random() < math.exp(-delta / max(temp, 1e-9))
        )

        if accept:
            current       = candidate
            current_score = cand_score
            current_metrics = cand_metrics

        if cand_score < best_score:
            best         = copy.deepcopy(candidate)
            best_score   = cand_score
            best_metrics = cand_metrics

        if it % 100 == 0 or it == 1:
            history.append({
                "iteration":   it,
                "temperature": temp,
                "best_score":  best_score,
                **best_metrics
            })

    return best, best_metrics, pd.DataFrame(history)


def local_search(best, data):
    current       = copy.deepcopy(best)
    current_score, current_metrics = evaluate_solution(current, data)
    improved      = True
    it            = 0
    history       = []

    while improved and it < LOCAL_SEARCH_ITERATIONS:
        improved = False
        it      += 1

        candidate_best       = current
        candidate_best_score = current_score
        candidate_best_metrics = current_metrics

        for _ in range(10):
            candidate = (
                targeted_overtime_mutation(current, data)
                if random.random() < 0.80
                else mutate_solution(current, data)
            )
            score, metrics = evaluate_solution(candidate, data)
            if score < candidate_best_score and metrics.get("feasible", False):
                candidate_best         = candidate
                candidate_best_score   = score
                candidate_best_metrics = metrics

        if candidate_best_score < current_score:
            current         = candidate_best
            current_score   = candidate_best_score
            current_metrics = candidate_best_metrics
            improved        = True

        if it % 20 == 0 or improved:
            history.append({
                "iteration": it,
                "best_score": current_score,
                **current_metrics
            })

    return current, current_metrics, pd.DataFrame(history)


# ============================================================
# 12. WARM START EXPORT
# ============================================================

def export_warm_start(final_sol, final_schedule, data):
    """
    Heuristic final solution'ını Gurobi warm start için JSON'a yazar.
    Exact model bu dosyayı okuyup model.optimize() öncesinde
    Gurobi değişkenlerine .Start atar.
    Gurobi'nin ilk Incumbent'ı = heuristic OT değeri olur → gap hesaplanabilir.
    """
    ps    = data["planning_start"]
    sched = final_schedule.copy()
    sched["start"]  = pd.to_datetime(sched["start"])
    sched["finish"] = pd.to_datetime(sched["finish"])

    warm = {
        "planning_start": ps.isoformat(),
        "tasks":  [],
        "splits": {},
        "groups": {},
        "qtys":   {},
    }

    for _, row in sched.iterrows():
        s_min = (row["start"]  - ps).total_seconds() / 60.0
        f_min = (row["finish"] - ps).total_seconds() / 60.0
        warm["tasks"].append({
            "job_id":      row["job_id"],
            "split_id":    int(row["split_id"]),
            "operation":   int(row["operation"]),
            "machine":     row["machine"],
            "start_min":   round(s_min, 4),
            "finish_min":  round(f_min, 4),
            "qty":         int(row["quantity"]),
            "group":       int(row["group"]),
            "overtime_min": float(row.get("overtime_min", 0.0)),
        })

    for job in data["jobs"]:
        jid = job.job_id
        qs  = final_sol["splits"].get(jid, [job.quantity])
        warm["splits"][jid] = [int(q) for q in qs]
        for sid, q in enumerate(qs, start=1):
            key = f"{jid}__{sid}"
            warm["groups"][key] = int(final_sol["group"].get((jid, sid), -1))
            warm["qtys"][key]   = int(q)

    with open(WARM_START_FILE, "w", encoding="utf-8") as f:
        json.dump(warm, f, ensure_ascii=False, indent=2)

    total_ot = sched["overtime_min"].sum()
    print(f"\nWarm start kaydedildi : {WARM_START_FILE}")
    print(f"  Heuristic OT        : {total_ot:.1f} dk")
    print(f"  Task sayısı         : {len(warm['tasks'])}")
    print(f"  Exact model bu değerden başlayacak → gap raporlanabilir.")


# ============================================================
# 13. OUTPUT FUNCTIONS
# ============================================================

def build_split_summary(schedule_df, data):
    df = add_split_count(schedule_df)
    jobs_by_id = {j.job_id: j for j in data["jobs"]}
    rows = []
    for job_id, g in df.groupby("job_id"):
        job = jobs_by_id[job_id]
        split_quantities = (
            g[["split_id", "quantity"]].drop_duplicates().sort_values("split_id")
            .apply(lambda r: f"S{int(r['split_id'])}={int(r['quantity'])}", axis=1).tolist()
        )
        rows.append({
            "job_id":            job_id,
            "part_no":           job.part_no,
            "original_quantity": job.quantity,
            "split_count":       int(g["split_count"].max()),
            "is_split":          "YES" if int(g["split_count"].max()) > 1 else "NO",
            "split_quantities":  ", ".join(split_quantities),
            "due_date":          job.due_date,
            "final_completion":  g[g["operation"] == 20]["finish"].max(),
        })
    return pd.DataFrame(rows).sort_values(["is_split", "due_date", "part_no"], ascending=[False, True, True])


def build_schedule_output(schedule_df):
    out = add_split_count(schedule_df).copy()
    out["start"]  = pd.to_datetime(out["start"])
    out["finish"] = pd.to_datetime(out["finish"])
    out["Tarih"]                       = out["start"].dt.date
    out["Başlangıç Saat"]              = out["start"].dt.strftime("%H:%M")
    out["Bitiş Saat"]                  = out["finish"].dt.strftime("%H:%M")
    out["Parça No"]                    = out["part_no"]
    out["Makine No"]                   = out["machine"]
    out["CNC-1 operasyonu(piston)"]    = np.where(out["operation"] == 10, "X", "")
    out["CNC-2 operasyonu (saplama)"]  = np.where(out["operation"] == 20, "X", "")
    out["Adet"]                        = out["quantity"]
    out["Çap"]                         = out["diameter"]
    out["Makine Grubu"]                = out["group"]
    out["Fazla mesai (dk)"]            = out["overtime_min"].round(1)
    out["Split"]                       = out["split_id"]
    out["Split Count"]                 = out["split_count"]
    out["Split Label"]                 = "S" + out["split_id"].astype(str) + "/" + out["split_count"].astype(str)
    out["Operation Label"]             = np.where(out["operation"] == 10, "Op10", "Op20")
    out["Setup Süresi (dk)"]           = out["setup_min"].round(1)
    out["İşlem Süresi (dk)"]           = out["processing_min"].round(1)
    out["Job ID"]                      = out["job_id"]

    cols = [
        "Tarih", "Başlangıç Saat", "Bitiş Saat", "Parça No", "Makine No",
        "CNC-1 operasyonu(piston)", "CNC-2 operasyonu (saplama)",
        "Adet", "Çap", "Makine Grubu", "Fazla mesai (dk)", "Split",
        "Split Count", "Split Label", "Operation Label",
        "Setup Süresi (dk)", "İşlem Süresi (dk)", "Job ID"
    ]
    return out[cols].sort_values(["Tarih", "Makine No", "Başlangıç Saat"]).reset_index(drop=True)


def build_machine_summary(schedule_df):
    df = schedule_df.copy()
    df["start"]  = pd.to_datetime(df["start"])
    df["finish"] = pd.to_datetime(df["finish"])
    summary = df.groupby("machine").agg(
        total_processing_min=("processing_min", "sum"),
        total_setup_min=("setup_min", "sum"),
        total_overtime_min=("overtime_min", "sum"),
        first_start=("start", "min"),
        last_finish=("finish", "max"),
        task_count=("job_id", "count"),
    ).reset_index()
    summary["total_load_min"] = summary["total_processing_min"] + summary["total_setup_min"]
    return summary.sort_values("total_overtime_min", ascending=False)


def build_job_summary(schedule_df, data):
    schedule_df           = schedule_df.copy()
    schedule_df["finish"] = pd.to_datetime(schedule_df["finish"])
    jobs_by_id            = {j.job_id: j for j in data["jobs"]}
    tardiness_limit_min   = TARDINESS_LIMIT_DAYS * 24 * 60
    comp  = schedule_df[schedule_df["operation"] == 20].groupby("job_id")["finish"].max().reset_index()
    rows  = []
    for _, r in comp.iterrows():
        job       = jobs_by_id[r["job_id"]]
        finish    = r["finish"]
        tardiness = max(0.0, (finish - job.due_date).total_seconds() / 60.0)
        rows.append({
            "job_id":                   job.job_id,
            "part_no":                  job.part_no,
            "quantity":                 job.quantity,
            "due_date":                 job.due_date,
            "completion":               finish,
            "tardiness_min":            tardiness,
            "tardiness_days":           tardiness / 1440.0,
            "allowed_latest":           job.due_date + timedelta(minutes=tardiness_limit_min),
            "tardiness_limit_violated": tardiness > tardiness_limit_min,
        })
    return pd.DataFrame(rows).sort_values(["due_date", "part_no"])


def create_excel_output(schedule_df, sa_history, ls_history, metrics, output_file, data):
    schedule_out    = build_schedule_output(schedule_df)
    machine_summary = build_machine_summary(schedule_df)
    sched_with_cnt  = add_split_count(schedule_df)
    job_summary     = build_job_summary(sched_with_cnt, data)
    split_summary   = build_split_summary(sched_with_cnt, data)
    overlap_check   = check_machine_overlaps(sched_with_cnt)
    if overlap_check.empty:
        overlap_check = pd.DataFrame([{"status": "NO OVERLAP — Feasible", "total_overlap_min": 0}])

    wb  = Workbook()
    ws1 = wb.active;                  ws1.title = "Optimized Schedule"
    ws2 = wb.create_sheet("Machine Summary")
    ws3 = wb.create_sheet("Job Summary")
    ws4 = wb.create_sheet("SA History")
    ws6 = wb.create_sheet("Split Summary")
    ws7 = wb.create_sheet("Overlap Check")

    def safe_value(v):
        if v is None: return ""
        if isinstance(v, float) and math.isinf(v): return "inf"
        if isinstance(v, (dict, list, set)): return str(v)
        if isinstance(v, bool): return str(v)
        return v

    def write_df(ws, df):
        df = df.copy()
        for col in df.columns:
            df[col] = df[col].apply(safe_value)
        ws.append(list(df.columns))
        for row in df.itertuples(index=False):
            ws.append([safe_value(v) for v in row])
        hf   = PatternFill("solid", fgColor="1F4E78")
        hfnt = Font(color="FFFFFF", bold=True)
        thin = Side(style="thin", color="D9E2F3")
        for cell in ws[1]:
            cell.fill      = hf
            cell.font      = hfnt
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border    = Border(bottom=thin)
        for col in range(1, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(col)].width = min(28, max(12, len(str(ws.cell(1, col).value)) + 4))
        ws.freeze_panes = "A2"

    write_df(ws1, schedule_out)
    write_df(ws2, machine_summary)
    write_df(ws3, job_summary)
    write_df(ws6, split_summary)
    write_df(ws7, overlap_check)

    metrics_clean = {
        k: (str(v) if isinstance(v, (dict, list, set)) else
            ("inf" if isinstance(v, float) and math.isinf(v) else v))
        for k, v in metrics.items()
    }
    all_cols  = list(pd.DataFrame([{"phase": "FINAL", **metrics_clean}]).columns)
    empty_row = pd.DataFrame([{c: "" for c in all_cols}])

    history_df = pd.concat([
        pd.DataFrame([{"phase": "FINAL", **metrics_clean}]),
        empty_row,
        sa_history.assign(phase="SA")              if not sa_history.empty  else pd.DataFrame(),
        ls_history.assign(phase="LOCAL_SEARCH")    if not ls_history.empty  else pd.DataFrame(),
    ], ignore_index=True)
    write_df(ws4, history_df)

    ws_kpi = wb.create_sheet("KPI")
    ws_kpi["A1"] = "OBJECTIVE FUNCTION: minimize Σm OTm"
    ws_kpi["A1"].font = Font(size=13, bold=True, color="FFFFFF")
    ws_kpi["A1"].fill = PatternFill("solid", fgColor="1F4E78")
    ws_kpi.merge_cells("A1:C1")

    kpi_rows = [
        ("--- ANA HEDEF ---", "", ""),
        ("Baseline Fazla Mesai (dk)",       metrics.get("baseline_overtime_min", 0.0),  "Σm OTm — sevkiyat verisi"),
        ("Optimize Fazla Mesai (dk)",       metrics["total_overtime"],                   "Σm OTm — heuristik çözüm"),
        ("Fazla Mesai İyileşmesi (dk)",     metrics.get("overtime_improvement_min", 0.0), "Baseline − Optimize"),
        ("Objective Score  f(s) = Σm OTm", metrics["score"],                            "= Optimize Fazla Mesai (dk)"),
        ("", "", ""),
        ("--- KISITLAR ---", "", ""),
        ("C1 Makine Çakışması",             metrics["overlap_count"],                    "0 olmalı"),
        ("C3 Gecikme Limiti (gün)",         TARDINESS_LIMIT_DAYS,                        "Hard constraint"),
        ("C3 Gecikme İhlali (dk)",          metrics["total_tardiness_limit_violation"],  "0 olmalı"),
        ("", "", ""),
        ("--- BİLGİ AMAÇLI ---", "", ""),
        ("Max Gecikme (gün)",               metrics["max_tardiness_days"],               "Raporlama"),
        ("Makespan (dk)",                   metrics["makespan"],                         "Raporlama"),
        ("Toplam Setup (dk)",               metrics["total_setup"],                      "Raporlama"),
    ]
    for i, (name, val, note) in enumerate(kpi_rows, start=2):
        ws_kpi.cell(i, 1).value = name
        ws_kpi.cell(i, 2).value = round(float(val), 3) if isinstance(val, (int, float)) else val
        ws_kpi.cell(i, 3).value = note
        ws_kpi.cell(i, 1).font  = Font(bold=("---" in str(name)))
    ws_kpi.column_dimensions["A"].width = 35
    ws_kpi.column_dimensions["B"].width = 20
    ws_kpi.column_dimensions["C"].width = 40

    wb.save(output_file)


# ============================================================
# 14. MAIN RUN
# ============================================================

if __name__ == "__main__":
    print("=" * 60)
    print("INITIAL SOLUTION CONSTRUCTION + WARM START EXPORT")
    print("SA/LS çalışmıyor — sadece initial solution Gurobi'ye verilir.")
    print(f"  C3 — Gecikme ≤ {TARDINESS_LIMIT_DAYS} gün ({TARDINESS_LIMIT_DAYS*24*60:.0f} dk)")
    print("=" * 60)

    print("\nLoading data...")
    GLOBAL_DATA = build_problem_data(
        shipment_file=SHIPMENT_FILE,
        sdst_file=SDST_FILE,
        machine_group_file=MACHINE_GROUP_FILE,
        use_shipped_quantity=True
    )
    print(f"Jobs: {len(GLOBAL_DATA['jobs'])} | Machines: {len(GLOBAL_DATA['machine_to_group'])}")

    print("\n--- Initial Solution Construction ---")
    initial_sol = build_initial_solution(GLOBAL_DATA)
    initial_sol = repair_solution(initial_sol, GLOBAL_DATA)

    init_score, init_metrics = evaluate_solution(initial_sol, GLOBAL_DATA)
    attempts = 0
    while init_score == INFEASIBLE_PENALTY and attempts < 500:
        initial_sol = repair_solution(build_initial_solution(GLOBAL_DATA), GLOBAL_DATA)
        init_score, init_metrics = evaluate_solution(initial_sol, GLOBAL_DATA)
        attempts += 1

    if init_score == INFEASIBLE_PENALTY:
        print("  HATA: 500 denemede feasible initial solution bulunamadı.")
        print("  Warm start oluşturulamadı.")
    else:
        initial_schedule = schedule_solution(initial_sol, GLOBAL_DATA)
        print(f"  Initial solution OT : {init_metrics['total_overtime']:.1f} dk")
        print(f"  Feasible            : {init_metrics.get('feasible', '?')}")
        print(f"  Deneme sayısı       : {attempts + 1}")
        print(f"\n  Bu değer Gurobi'nin ilk Incumbent'ı olacak.")
        print(f"  Gurobi 6 saat sonunda gap = (Incumbent - BestBound) / Incumbent")

        print("\nExporting warm start...")
        export_warm_start(initial_sol, initial_schedule, GLOBAL_DATA)

        print(f"\n{'='*60}")
        print(f"SONUÇ")
        print(f"  Initial OT (Gurobi Incumbent) : {init_metrics['total_overtime']:.1f} dk")
        print(f"  Warm start dosyası            : {WARM_START_FILE}")
        print(f"  Şimdi exact modeli çalıştırın.")
        print(f"{'='*60}")

Using files:
SHIPMENT_FILE      = C:\Users\emrec\Documents\parallel_scheduling\heuristicandexact\492-güncel sevkiyat.xlsx
SDST_FILE          = C:\Users\emrec\Documents\parallel_scheduling\heuristicandexact\SDST.xlsx
MACHINE_GROUP_FILE = C:\Users\emrec\Documents\parallel_scheduling\heuristicandexact\machine_group_data.xlsx
INITIAL SOLUTION CONSTRUCTION + WARM START EXPORT
SA/LS çalışmıyor — sadece initial solution Gurobi'ye verilir.
  C3 — Gecikme ≤ 1.5 gün (2160 dk)

Loading data...
Duplicate order rows aggregated: 28 -> 27 unique jobs
Jobs: 27 | Machines: 12

--- Initial Solution Construction ---
  Initial solution OT : 3256.9 dk
  Feasible            : True
  Deneme sayısı       : 1

  Bu değer Gurobi'nin ilk Incumbent'ı olacak.
  Gurobi 6 saat sonunda gap = (Incumbent - BestBound) / Incumbent

Exporting warm start...

Warm start kaydedildi : C:\Users\emrec\Documents\parallel_scheduling\heuristicandexact\heuristic_warm_start.json
  Heuristic OT        : 3256.9 dk
  Task sayısı       